# 47. Low-level point-to-point dissimilarity (PPD)

**Objectives:**
- Run `point_to_point_dissimilarity` directly on plain coordinate/density arrays (no
  `FitSession`), following M. Williams, "How good are your fits?" (arXiv:1006.3019), Sec. 3.3.
- Compare a toy sample against an independent toy from the *same* model: expect a
  non-significant p-value.
- Compare it against a toy from a *visibly different* model: expect a small/significant p-value.

`n_permutations`/sample sizes are kept small so this runs quickly; see `docs/goodness_of_fit.md`
for the session-level convenience wrapper (`FitSession.point_to_point_dissimilarity`) that builds
these same arrays from a real fit. Run cells in order in a fresh kernel.


In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import numpy as np

from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, Parameter, RealImag, Resonance,
    generate_toy, point_to_point_dissimilarity,
)

## 1. Model A (the null hypothesis `f0`) and a visibly different Model B

Model A is the rho(770) + non-resonant model used elsewhere in this course. Model B keeps the
same channel and normalization but drops the resonance entirely (a flat non-resonant-only
amplitude) -- a clearly different Dalitz-plot shape.

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
x = Parameter.coefficient("NR.x", 0.55, owner="NR", bounds=(-2, 2), step=0.02)
y = Parameter.coefficient("NR.y", 0.30, owner="NR", bounds=(-2, 2), step=0.02)

model_a = DecayModel(
    channel,
    [
        Resonance("rho", (0, 1), RealImag(1, 0), mass=0.7753, width=0.1491, spin=1),
        NonResonant(RealImag(x, y), name="NR"),
    ],
    normalization_method="square-dalitz", normalization_resolution=60,
    normalization_pair=(0, 1),
)
model_b = DecayModel(
    channel,
    [NonResonant(RealImag(1.0, 0.0), name="NR")],
    normalization_method="square-dalitz", normalization_resolution=60,
    normalization_pair=(0, 1),
)
truth_a = {p.name: p.value for p in model_a.parameters}
truth_b = {p.name: p.value for p in model_b.parameters}

f0 = model_a.pdf()  # the density every comparison below is tested against
phase_space_area = float(np.mean(np.asarray(model_a.normalization_sample.weights)))
print(f"phase_space_area = {phase_space_area:.4f}")

phase_space_area = 5.0631


## 2. Three independent toys

`toy_a1` plays the role of "data"; `toy_a2` is an independent replica from the *same* model
(should agree with `f0`); `toy_b` is a replica from the *visibly different* model (should not).

In [3]:
toy_a1 = generate_toy(model_a, 300, parameters=truth_a, seed=101, inverse_resolution=256)
toy_a2 = generate_toy(model_a, 1200, parameters=truth_a, seed=202, inverse_resolution=256)
toy_b = generate_toy(model_b, 1200, parameters=truth_b, seed=303, inverse_resolution=256)

def xy(sample):
    return np.column_stack([np.asarray(sample.s12), np.asarray(sample.s13)])

def density(sample):
    return np.asarray(f0(sample.as_dict(), truth_a))

xy_a1, density_a1 = xy(toy_a1), density(toy_a1)
xy_a2, density_a2 = xy(toy_a2), density(toy_a2)
xy_b, density_b = xy(toy_b), density(toy_b)

## 3. Same-model comparison: expect a non-significant p-value

`data_xy`/`reference_xy` are `toy_a1`/`toy_a2`; both are independent draws from `f0` itself, so
the observed statistic `T` should sit comfortably inside the null distribution the permutation
test builds from the pooled sample.

In [4]:
result_same = point_to_point_dissimilarity(
    xy_a1, xy_a2, density_a1, density_a2,
    sigma_bar=0.02, phase_space_area=phase_space_area,
    n_permutations=100, seed=1, max_total_events=6_000,
)
print(f"same-model:      T={result_same.statistic:.6g}  p-value={result_same.p_value:.3f}")
print(f"  n_data={result_same.n_data}  n_reference={result_same.n_reference}")

same-model:      T=-0.000237635  p-value=0.330
  n_data=300  n_reference=1200


## 4. Different-model comparison: expect a small/significant p-value

Same `data_xy` (`toy_a1`), but `reference_xy` is now `toy_b`, drawn from the resonance-free
model. `sigma(x)` still uses `f0` (model A's density) in both cases -- the *test* is always "does
this data look like `f0`", not "do the two point clouds match each other" -- so the comparison
stays apples-to-apples with Sec. 3.

In [5]:
result_diff = point_to_point_dissimilarity(
    xy_a1, xy_b, density_a1, density_b,
    sigma_bar=0.02, phase_space_area=phase_space_area,
    n_permutations=100, seed=1, max_total_events=6_000,
)
print(f"different-model:  T={result_diff.statistic:.6g}  p-value={result_diff.p_value:.3f}")
print(f"  n_data={result_diff.n_data}  n_reference={result_diff.n_reference}")

assert result_diff.p_value < result_same.p_value

different-model:  T=-0.000734948  p-value=0.000
  n_data=300  n_reference=1200


## Summary and exercises

1. Raise `n_permutations` to a few hundred and see the same-model p-value settle rather than
   jumping between coarse quantiles -- with 100 permutations, `p_value` can only take multiples
   of 0.01.
2. Try a smaller model mismatch (e.g. keep the rho but change `NR.x`/`NR.y` by only 20%) and see
   `result_diff.p_value` move partway back toward `result_same.p_value`.
3. Recompute `density_a1`/`density_a2`/`density_b` using `model_b.pdf()` instead of `f0` (testing
   the *wrong* null hypothesis against `toy_b`-like data) and see the same-model/different-model
   roles trade places.

Reference: `docs/goodness_of_fit.md` ("Point-to-point dissimilarity (PPD)"), M. Williams,
arXiv:1006.3019.

Return to [the course guide](TUTORIALS.md).
